# google_cloud_bigquery_storage smoke test

Smoke test for `google-cloud-bigquery-storage`, the client for the BigQuery
Storage Read and Write APIs. Every example runs offline: it builds request
messages, round-trips them through their protobuf wire format, uses the
resource-path helpers, and constructs clients with anonymous credentials.
No request is sent to Google Cloud.

## Example 1: ReadSession message round trip

In [ ]:
from google.cloud.bigquery_storage_v1 import types

session = types.ReadSession(
    table="projects/p/datasets/d/tables/t",
    data_format=types.DataFormat.ARROW,
    read_options=types.ReadSession.TableReadOptions(
        selected_fields=["a", "b"], row_restriction="a > 0"
    ),
)
wire = types.ReadSession.serialize(session)
print("serialized ReadSession:", len(wire), "bytes")
back = types.ReadSession.deserialize(wire)
assert back == session
assert back.data_format == types.DataFormat.ARROW
assert list(back.read_options.selected_fields) == ["a", "b"]
assert back.read_options.row_restriction == "a > 0"
print("ReadSession round trip OK")

## Example 2: Resource path helpers

In [ ]:
from google.cloud.bigquery_storage_v1 import BigQueryReadClient, BigQueryWriteClient

session_path = BigQueryReadClient.read_session_path("p", "l", "s")
assert session_path == "projects/p/locations/l/sessions/s"
assert BigQueryReadClient.parse_read_session_path(session_path) == {
    "project": "p", "location": "l", "session": "s",
}
assert BigQueryReadClient.read_stream_path("p", "l", "s", "st") == "projects/p/locations/l/sessions/s/streams/st"
assert BigQueryReadClient.table_path("p", "d", "t") == "projects/p/datasets/d/tables/t"
assert BigQueryWriteClient.write_stream_path("p", "d", "t", "w") == "projects/p/datasets/d/tables/t/streams/w"
print("read session path:", session_path)
print("path helpers OK")

## Example 3: Client construction with anonymous credentials

In [ ]:
from google.auth.credentials import AnonymousCredentials
from google.cloud.bigquery_storage_v1 import BigQueryReadClient, BigQueryWriteClient

# Building a client opens no connection; RPCs only start when a method is called.
read_client = BigQueryReadClient(credentials=AnonymousCredentials())
write_client = BigQueryWriteClient(credentials=AnonymousCredentials())
print("read transport:", type(read_client.transport).__name__)
print("write transport:", type(write_client.transport).__name__)
assert read_client.api_endpoint == "bigquerystorage.googleapis.com"
assert write_client.api_endpoint == "bigquerystorage.googleapis.com"
print("clients constructed")

## Example 4: CreateReadSessionRequest as JSON

In [ ]:
from google.cloud.bigquery_storage_v1 import types

request = types.CreateReadSessionRequest(
    parent="projects/p",
    read_session=types.ReadSession(
        table="projects/p/datasets/d/tables/t", data_format=types.DataFormat.AVRO
    ),
    max_stream_count=1,
)
as_json = types.CreateReadSessionRequest.to_json(request)
assert '"maxStreamCount": 1' in as_json and '"parent": "projects/p"' in as_json
assert types.CreateReadSessionRequest.from_json(as_json) == request
as_dict = types.CreateReadSessionRequest.to_dict(request)
assert as_dict["read_session"]["table"] == "projects/p/datasets/d/tables/t"
assert as_dict["read_session"]["data_format"] == types.DataFormat.AVRO
print("JSON and dict round trip OK")

## Example 5: AppendRowsRequest with a protobuf writer schema

In [ ]:
from google.cloud.bigquery_storage_v1 import types
from google.protobuf import descriptor_pb2

# The Write API sends rows as serialized protobuf messages plus the message's descriptor.
row_type = descriptor_pb2.DescriptorProto(name="Row")
row_type.field.add(
    name="id", number=1,
    type=descriptor_pb2.FieldDescriptorProto.TYPE_INT64,
    label=descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL,
)
request = types.AppendRowsRequest(
    write_stream="projects/p/datasets/d/tables/t/streams/_default",
    proto_rows=types.AppendRowsRequest.ProtoData(
        writer_schema=types.ProtoSchema(proto_descriptor=row_type),
        rows=types.ProtoRows(serialized_rows=[b"\x08\x07"]),  # field 1 = 7
    ),
)
back = types.AppendRowsRequest.deserialize(types.AppendRowsRequest.serialize(request))
assert types.AppendRowsRequest.pb(back).WhichOneof("rows") == "proto_rows"
assert back.proto_rows.writer_schema.proto_descriptor.field[0].name == "id"
assert list(back.proto_rows.rows.serialized_rows) == [b"\x08\x07"]

schema = types.TableSchema(fields=[types.TableFieldSchema(
    name="id", type_=types.TableFieldSchema.Type.INT64, mode=types.TableFieldSchema.Mode.REQUIRED,
)])
assert schema.fields[0].type_.name == "INT64" and schema.fields[0].mode.name == "REQUIRED"
print("AppendRowsRequest round trip OK")